# Educational Disease–Symptom Subset

Reads `disease_symptom_importance.csv` and filters it down to **10 representative diseases**
and **10 discriminating symptoms** suitable for teaching differential diagnosis.

---

## Rationale

### Diseases selected (diverse body systems & clinical contexts)

| Disease | System / Context |
|---|---|
| Pneumonia | Respiratory / infectious |
| Migraine | Neurological |
| Heart attack | Cardiovascular / emergency |
| Appendicitis | Acute abdomen / surgical |
| Anaphylaxis | Allergic / emergency |
| Major depression | Psychiatric |
| Stroke | Neurological / vascular |
| Urinary tract infection | Urological / infectious |
| Gout | Rheumatological / metabolic |
| Meningitis | Infectious / neurological |

### Symptoms selected (maximally discriminating across the 10 diseases)

| Symptom | Best distinguishes |
|---|---|
| Fever | Meningitis, pneumonia, appendicitis |
| Headache | Migraine, meningitis |
| Coughing | Pneumonia |
| Seizures | Stroke, meningitis |
| Depression | Major depression |
| Pain during urination | UTI |
| Shortness of breath | Heart attack, anaphylaxis |
| Swelling | Gout, anaphylaxis |
| Sensitivity to light | Migraine, meningitis |
| Stiffness | Meningitis |

In [1]:
import pandas as pd

# ── 1. Load the full tidy CSV ─────────────────────────────────────────────────
INPUT_FILE  = 'disease_symptom_importance.csv'   # adjust path if needed
OUTPUT_FILE = 'educational_subset.csv'

df = pd.read_csv(INPUT_FILE)
print(f'Loaded {len(df):,} rows | {df["disease"].nunique()} diseases | {df["symptom"].nunique()} symptoms')
df.head()

Loaded 3,709 rows | 156 diseases | 330 symptoms


,disease,symptom,importance
0,abscess,pain,0.318
1,abscess,fever,0.119
2,abscess,swelling,0.112
3,abscess,redness,0.094
4,abscess,chills,0.092


In [2]:
# ── 2. Define the 10 representative diseases ──────────────────────────────────
DISEASES = [
    'pneumonia',
    'migraine',
    'heart attack',
    'appendicitis',
    'anaphylaxis',
    'major depression',
    'stroke',
    'urinary tract infection',
    'gout',
    'meningitis',
]

# ── 3. Define the 10 discriminating symptoms ──────────────────────────────────
SYMPTOMS = [
    'fever',
    'headache',
    'coughing',
    'seizures',
    'depression',
    'pain during urination',
    'shortness of breath',
    'swelling',
    'sensitivity to light',
    'stiffness',
]

print(f'Filtering to {len(DISEASES)} diseases x {len(SYMPTOMS)} symptoms...')

Filtering to 10 diseases x 10 symptoms...


In [3]:
# ── 4. Apply the filter ───────────────────────────────────────────────────────
df_filtered = df[
    df['disease'].isin(DISEASES) &
    df['symptom'].isin(SYMPTOMS)
].copy()

# Preserve a predictable ordering: diseases in DISEASES order, symptoms in SYMPTOMS order
df_filtered['disease'] = pd.Categorical(df_filtered['disease'], categories=DISEASES, ordered=True)
df_filtered['symptom'] = pd.Categorical(df_filtered['symptom'], categories=SYMPTOMS, ordered=True)
df_filtered = df_filtered.sort_values(['disease', 'symptom']).reset_index(drop=True)

print(f'Result: {len(df_filtered)} (disease, symptom) pairs')
df_filtered

Result: 31 (disease, symptom) pairs


,disease,symptom,importance
0,pneumonia,fever,0.208
1,pneumonia,coughing,0.402
2,pneumonia,shortness of breath,0.156
3,migraine,headache,0.384
4,migraine,seizures,0.017
5,migraine,depression,0.026
6,migraine,sensitivity to light,0.223
7,migraine,stiffness,0.013
8,heart attack,depression,0.020
9,heart attack,shortness of breath,0.041


In [4]:
# ── 5. Sanity check: verify coverage ─────────────────────────────────────────
# Show which (disease, symptom) pairs are MISSING (importance = 0 / not in data)
from itertools import product

all_pairs = pd.DataFrame(list(product(DISEASES, SYMPTOMS)), columns=['disease', 'symptom'])
merged    = all_pairs.merge(df_filtered, on=['disease', 'symptom'], how='left')
missing   = merged[merged['importance'].isna()]

print(f'Pairs present : {len(df_filtered)}')
print(f'Pairs absent  : {len(missing)}  (disease has 0 recorded importance for that symptom)')
print()

# Pivot table for a quick visual overview
pivot = merged.pivot_table(
    index='disease', columns='symptom', values='importance', fill_value=0.0
)[SYMPTOMS].round(3)
pivot.index = pd.CategoricalIndex(pivot.index, categories=DISEASES, ordered=True)
pivot = pivot.sort_index()
print('Importance matrix (0.0 = symptom not recorded for that disease):')
pivot

Pairs present : 31
Pairs absent  : 69  (disease has 0 recorded importance for that symptom)

Importance matrix (0.0 = symptom not recorded for that disease):


symptom,fever,headache,coughing,seizures,depression,pain during urination,shortness of breath,swelling,sensitivity to light,stiffness
disease,,,,,,,,,,
pneumonia,0.208,0.000,0.402,0.000,0.000,0.000,0.156,0.000,0.000,0.000
migraine,0.000,0.384,0.000,0.017,0.026,0.000,0.000,0.000,0.223,0.013
heart attack,0.000,0.000,0.000,0.000,0.020,0.000,0.041,0.000,0.000,0.000
appendicitis,0.096,0.000,0.000,0.000,0.000,0.022,0.000,0.000,0.000,0.000
anaphylaxis,0.000,0.000,0.000,0.020,0.000,0.000,0.099,0.154,0.000,0.000
major depression,0.000,0.008,0.000,0.026,0.416,0.009,0.000,0.000,0.000,0.000
stroke,0.000,0.076,0.000,0.044,0.000,0.000,0.000,0.000,0.000,0.000
urinary tract infection,0.071,0.000,0.000,0.000,0.000,0.146,0.000,0.000,0.000,0.000
gout,0.011,0.000,0.000,0.000,0.000,0.000,0.000,0.246,0.000,0.000


In [5]:
# ── 6. Save the filtered CSV ──────────────────────────────────────────────────
# Reset Categorical dtype back to plain strings before saving
df_out = df_filtered.copy()
df_out['disease'] = df_out['disease'].astype(str)
df_out['symptom'] = df_out['symptom'].astype(str)

df_out.to_csv(OUTPUT_FILE, index=False)
print(f'Saved -> {OUTPUT_FILE}  ({len(df_out)} rows)')

Saved -> educational_subset.csv  (31 rows)
